In [19]:
## Imports from activity 6: apis_blank
import pandas as pd
import numpy as np
import re
import requests
import yaml
import os

## Imports for wikipedia pulls
import pywikibot as pw
import time
pw.config.user_agent = 'EskildsenLogan research project (logan.j.eskildsen.28@dartmouth.edu) Pywikibot/11.6.0'
site = pw.Site("en", "wikipedia")
headers = {
    'User-Agent': 'EskildsenLogan research project (logan.j.eskildsen.28@dartmouth.edu)'
}

## Imports for test visualization, if needed
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import curve_fit

## Imports for google search trends
!pip install pytrends
from pytrends.request import TrendReq

## Set font parameters, if needed
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]


## Repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [85]:
## All functions located here

## Function 1: Iterates over the country column in basic_info df, gets all data on subcategories for each country that US played.
def get_wikipedia_subcategories(countries):
    ## For loop to pull each category associated with each country
    for cat_name in countries:
        
        cat = pw.Category(site, cat_name)

        # Use try-except to get information on pages per subcategory
        for subcat in cat.subcategories():

            try:

                all_info = subcat.categoryinfo

                num_articles = all_info.get('pages', 0)

                cat_title = subcat.title()

                subcategory_data.append({
                    "country": cat_name,
                    "subcategory_title": cat_title,
                    "article_count": num_articles
                })
                ## Need time.sleep() because I kept getting 429 errors, making too many requests.
                time.sleep(2)

            except KeyError:

                 print(f"{subcat.title()}: 0 articles (or info unavailable)")

## Function 2: Iterates over rows in the subcategory_info_final dataframe to get all article title data within each subcategory.
def get_wikipedia_articles(subcategory_info_final):
    for _, row in subcategory_info_final.iterrows():
        country = row['country']
        subcat_title = row['subcategory_title'] 
        subcat = pw.Category(site, subcat_title)

        ## Routinely getting errors from the pull, use the try, except logic to resolve issues
        try:
            for page in subcat.articles():
                article_data.append({'country': country,
                    'subcategory_title_fixed': row['subcategory_title_fixed'],
                    'article_title': page.title()
                })
            time.sleep(4)  
        except Exception as e:
            print(f"{e}")

## Function 3: Iterates over rows in the article_info_df dataframe to get all pageview data within each article from 5/17/2026 - 8/20/2026
def get_wikipedia_pageviews(article_info_df, cup_windows, headers):
    for _, row in article_info_df.iterrows():
        country = row['country']
        subcat_title = row['subcategory_title_fixed']
        article_title = row['article_title'].replace(' ', '_')
        
        start_date, end_date = cup_windows[country][0]

        ## Makes a direct query to the api instead of using built in pywikibot methods.
        wikipedia_api_query = (
            'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
            f'en.wikipedia.org/all-access/all-agents/{article_title}/daily/{start_date}/{end_date}'
        )
        
        try:
            wikipedia_api_query = requests.get(wikipedia_api_query, headers=headers)
            ## items is the key to the pulled data that I got with .key() method
            items = wikipedia_api_query.json().get('items', [])
            if wikipedia_api_query.status_code != 200:
                print(country, article_title, wikipedia_api_query.status_code, wikipedia_api_query.text[:300])
            
            for item in items:
                article_daily_data.append({
                    'country': country,
                     'subcategory_title_fixed': subcat_title,
                    'article_title': article_title,
                    'date': item['timestamp'][:8],  # YYYYMMDD
                    'views': item.get('views', 0)
                })
        except Exception as e:
            print(f"{e}")
        
        time.sleep(0.05)
    
    return pd.DataFrame(article_daily_data)

## Function 4: Exact same as function 3, except a query for the country's mainpage to compare as a control.
def get_country_wikipedia_pageviews(cup_windows, headers):
    
    for country, windows in cup_windows.items():
        
        for start_date, end_date in windows:
            pv_query = (
                'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
                f'en.wikipedia.org/all-access/all-agents/{country}/daily/{start_date}/{end_date}'
            )
            
            try:
                pv_resp = requests.get(pv_query, headers=headers)
                items = pv_resp.json().get('items', [])
                for item in items:
                    country_view_data.append({
                        'country': country,
                        'article_title': country,
                        'date': item['timestamp'][:8],
                        'views': item.get('views', 0)
                      })
                    
            except Exception as e:
                print(f"{e}")
            
            time.sleep(0.05)
    
    return pd.DataFrame(country_view_data)

## Function 5: Exact same purpose as function 4, just for getting the national football team in a separate dataset.
def get_football_wikipedia_pageviews(cup_windows, headers):
    
    for country, windows in cup_windows.items():
        
        for start_date, end_date in windows:
            pv_query = (
                'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
                f'en.wikipedia.org/all-access/all-agents/{country}_national_football_team/daily/{start_date}/{end_date}'
            )
            
            try:
                pv_resp = requests.get(pv_query, headers=headers)
                items = pv_resp.json().get('items', [])
                for item in items:
                    country_football_data.append({
                        'country': country,
                        'article_title': country + ' National Football Team',
                        'date': item['timestamp'][:8],
                        'views': item.get('views', 0)
                      })
                    
            except Exception as e:
                print(f"{e}")
            
            time.sleep(0.05)
    
    return pd.DataFrame(country_football_data)

## Function 6: gathers timeseries of search proportion between match and culture using keywords for each of the 11 host cities
def regional_search_interest(cup_windows, host_codes):

    
    for country, windows in cup_windows.items():
        for start, end in windows:
            start_str = str(start)
            end_str = str(end)

            ## Have to rebuild dates in string format for trends API to read properly. Easy with regex
            start_fmt = f"{start_str[:4]}-{start_str[4:6]}-{start_str[6:]}"
            end_fmt = f"{end_str[:4]}-{end_str[4:6]}-{end_str[6:]}"
            timeframe = f"{start_fmt} {end_fmt}"

            ## Define two terms for culture versus match data, aggregate later in 1_clean_merge_data.ipynb.
            keyword_list = [f"{country.replace('_', ' ')} history", f"{country.replace('_', ' ')} culture",
                            f"{country.replace('_', ' ')} World Cup", f"{country.replace('_', ' ')} soccer"]
            for code in host_codes:
    
                ## This is where I needed help with specifying keyworks, and above with defining the timeline.
                try:
                    pytrend.build_payload(kw_list=keyword_list, timeframe=timeframe, geo=code)
                    interest_df = pytrend.interest_over_time()
                    interest_df = interest_df.reset_index().rename(columns={f"{country.replace('_', ' ')} history": 'cultural_interest_1',
                                                                           f"{country.replace('_', ' ')} culture": 'cultural_interest_2',
                                                                           f"{country.replace('_', ' ')} World Cup": 'match_interest_1',
                                                                           f"{country.replace('_', ' ')} soccer": 'match_interest_2'})
                
                    interest_df['country'] = country
                    interest_df['geo_code'] = code 
                    regional_search_results.append(interest_df)
                except Exception as e:
                    print(f"{country} {code} {e}")
                    
                time.sleep(10)
    
    return pd.concat(regional_search_results, ignore_index=True)

In [17]:
## Part 1: creates foundational dataframe for project, scraped from google searches

## List of countries U.S. has played since 2014. Include year, match outcome, match score, match date, match type.
country_list = ["Ghana", "Portugal", "Germany", "Belgium",
                "Wales", "England", "Iran", "Netherlands", 
                "Paraguay", "Australia", "Turkey", "Bosnia_and_Herzegovina", "Belgium"]

year_played = [2014, 2014, 2014, 2014,
               2022, 2022, 2022, 2022, 
               2026, 2026, 2026, 2026, 2026]

match_outcome = ["win", "draw", "loss", "loss",
                 "draw", "draw", "win", "loss", 
                 "win", "win", "loss", "win", "loss"]

match_score = ["2-1", "2-2", "0-1", "1-2",
              "1-1", "0-0", "1-0", "1-3",
              "4-1", "2-0", "2-3", "2-0", "1-4"]

## Find the score differential with regex, kind of fun but super unneccessary

score_pattern = r'(\d)-(\d)'

score_differential = []

for score in match_score:
    match = re.match(score_pattern, score)
    if match:
        score_differential.append(int(match.group(1)) - int(match.group(2)))

match_date = ["20140614", "20140622", "20140626", "20140701",
             "20221121", "20221125", "20221129", "20221203",
             "20260612", "20260619", "20260625", "20260701", "20260706"]

match_type = ["group", "group", "group", "knockout",
            "group", "group", "group", "knockout",
            "group", "group", "group", "knockout", "knockout"]

## Create dataframe for visualization
basic_info_df = pd.DataFrame({"country": country_list, "year": year_played, 
                           "match_date": match_date, "outcome": match_outcome, "match_type": match_type,
                              "score": match_score, "score_differential": score_differential})

## Originally intended for this project to focus on 2014-2026 data. The scope was too large. Filter down to 2026.

basic_info_df = basic_info_df[basic_info_df["year"] == 2026].reset_index(drop=True)

basic_info_df


## Now store the data as a csv to call later so I do not have to rerun code.
basic_info_df.to_csv('../data/basic_info.csv', index=False)

,country,year,match_date,outcome,match_type,score,score_differential
0,Paraguay,2026,20260612,win,group,4-1,3
1,Australia,2026,20260619,win,group,2-0,2
2,Turkey,2026,20260625,loss,group,2-3,-1
3,Bosnia_and_Herzegovina,2026,20260701,win,knockout,2-0,2
4,Belgium,2026,20260706,loss,knockout,1-4,-3


In [21]:
## Part 2: Access popular article data
## Use pywikibot library to extract main Wiki subcategories for each country category. List of countries defined in basic_info_df.

## Empty list to store data with Function 1.
subcategory_data = []

## Call Function 1, create dataframe from subcat_data list.
get_wikipedia_subcategories(basic_info["country"])

subcategory_info_df = pd.DataFrame(subcategory_data)

subcategory_info_df

,country,subcategory_title,article_count
0,Paraguay,Category:Paraguay-related lists,4
1,Paraguay,Category:Buildings and structures in Paraguay,3
2,Paraguay,Category:Culture of Paraguay,9
3,Paraguay,Category:Economy of Paraguay,12
4,Paraguay,Category:Education in Paraguay,2
...,...,...,...
81,Belgium,Category:Belgian people,2
82,Belgium,Category:Politics of Belgium,40
83,Belgium,Category:Society of Belgium,12
84,Belgium,Category:Images of Belgium,1


In [22]:
## Removes stubs from subcategories since it confounds data
subcategory_info_final = subcategory_info_df[~subcategory_info_df['subcategory_title'].str.contains('stubs', na=False)].copy()

## Format the data better by removing "Category:" using the regex skills we learned in class
subcategory_info_final["subcategory_title_fixed"] = subcategory_info_final["subcategory_title"].str.replace('^Category:', '', regex=True)

## Now display presentable dataframe
subcategory_info_df_final = subcategory_info_final[["country", "subcategory_title_fixed", "subcategory_title", "article_count"]]

subcategory_info_df_final

subcategory_info_df_final.to_csv("../data/subcategory_data.csv", index=False)

,country,subcategory_title_fixed,subcategory_title,article_count
0,Paraguay,Paraguay-related lists,Category:Paraguay-related lists,4
1,Paraguay,Buildings and structures in Paraguay,Category:Buildings and structures in Paraguay,3
2,Paraguay,Culture of Paraguay,Category:Culture of Paraguay,9
3,Paraguay,Economy of Paraguay,Category:Economy of Paraguay,12
4,Paraguay,Education in Paraguay,Category:Education in Paraguay,2
...,...,...,...,...
80,Belgium,Organisations based in Belgium,Category:Organisations based in Belgium,21
81,Belgium,Belgian people,Category:Belgian people,2
82,Belgium,Politics of Belgium,Category:Politics of Belgium,40
83,Belgium,Society of Belgium,Category:Society of Belgium,12


In [34]:
## Info on DataFrame, they are the same but I didn't want to overwrite the other.
subcategory_info_final.info
subcategory_info_df_final.info

<bound method DataFrame.info of      country                              subcategory_title  article_count  \
0   Paraguay                Category:Paraguay-related lists              4   
1   Paraguay  Category:Buildings and structures in Paraguay              3   
2   Paraguay                   Category:Culture of Paraguay              9   
3   Paraguay                   Category:Economy of Paraguay             12   
4   Paraguay                 Category:Education in Paraguay              2   
..       ...                                            ...            ...   
80   Belgium        Category:Organisations based in Belgium             21   
81   Belgium                        Category:Belgian people              2   
82   Belgium                   Category:Politics of Belgium             40   
83   Belgium                    Category:Society of Belgium             12   
84   Belgium                     Category:Images of Belgium              1   

                 subcategory_ti

<bound method DataFrame.info of      country               subcategory_title_fixed  \
0   Paraguay                Paraguay-related lists   
1   Paraguay  Buildings and structures in Paraguay   
2   Paraguay                   Culture of Paraguay   
3   Paraguay                   Economy of Paraguay   
4   Paraguay                 Education in Paraguay   
..       ...                                   ...   
80   Belgium        Organisations based in Belgium   
81   Belgium                        Belgian people   
82   Belgium                   Politics of Belgium   
83   Belgium                    Society of Belgium   
84   Belgium                     Images of Belgium   

                                subcategory_title  article_count  
0                 Category:Paraguay-related lists              4  
1   Category:Buildings and structures in Paraguay              3  
2                    Category:Culture of Paraguay              9  
3                    Category:Economy of Paraguay  

In [24]:
## Define a function that pulls articles from each subcategory. First outline dates defined in window of analysis.
start_date_2026 = 20260517
end_date_2026 = 20260820

## Create dictionary of start and end dates for each country. 
## NOTE: I understand this is INCREDIBLY unintuitive. I originally wrote this code to account for the U.S. matching up against Belgium in 2014 and 2026.
## This is why I created a dictionary with a list of tuples. Really sad that MediaWiki API data does not cover anything before 2015. Please forgive me.
## If I were to do this again, I would probably just add start and end windows to basic_info_df. Too lazy

cup_windows = {
    'Australia': [(start_date_2026, end_date_2026)],
    'Belgium': [(start_date_2026, end_date_2026)],  # or handle 2014/2022 separately if kept distinct
    'Bosnia_and_Herzegovina': [(start_date_2026, end_date_2026)],
    'Paraguay': [(start_date_2026, end_date_2026)],
    'Turkey': [(start_date_2026, end_date_2026)],
}

In [41]:
## Now pull article data for each subcategory, long run time.

## Storing data outside the loop as requested in the project guidelines.
article_data = []

## Now call function 2, unravels the nest a bit more.
get_wikipedia_articles(subcategory_info_final)
article_info_df = pd.DataFrame(article_data)


KeyboardInterrupt



In [36]:
## Visualize DataFrame, save as .csv
article_info_df
article_info_df.info

,country,subcategory_title_fixed,article_title
0,Paraguay,Paraguay-related lists,Outline of Paraguay
1,Paraguay,Paraguay-related lists,International rankings of Paraguay
2,Paraguay,Paraguay-related lists,List of festivals in Paraguay
3,Paraguay,Paraguay-related lists,List of Paraguayan flags
4,Paraguay,Buildings and structures in Paraguay,Icono Tower
...,...,...,...
1205,Belgium,Society of Belgium,Ondertrouw
1206,Belgium,Society of Belgium,Unionism in Belgium
1207,Belgium,Images of Belgium,Leo Belgicus
1208,Belgium,Images of Belgium,File:Belgian Federal Science Policy Office log...


<bound method DataFrame.info of        country               subcategory_title_fixed  \
0     Paraguay                Paraguay-related lists   
1     Paraguay                Paraguay-related lists   
2     Paraguay                Paraguay-related lists   
3     Paraguay                Paraguay-related lists   
4     Paraguay  Buildings and structures in Paraguay   
...        ...                                   ...   
1205   Belgium                    Society of Belgium   
1206   Belgium                    Society of Belgium   
1207   Belgium                     Images of Belgium   
1208   Belgium                     Images of Belgium   
1209   Belgium                     Images of Belgium   

                                          article_title  
0                                   Outline of Paraguay  
1                    International rankings of Paraguay  
2                         List of festivals in Paraguay  
3                              List of Paraguayan flags  
4    

In [44]:
## Now define a user-generated function to call daily article views for the cup windows we defined earlier

article_daily_data = []

## Call function 3 to get views for each article title from 5/17/2026-8/20/2026, 1209*95*0.05=5900 second runtime, surely not.
article_views_df = get_wikipedia_pageviews(article_info_df, cup_windows, headers)
article_views_df

Australia AC/DC 404 {"detail":"invalid route","method":"get","status":404,"title":"Not Found","type":"about:blank","uri":"/metrics/pageviews/per-article/en.wikipedia.org/all-access/all-agents/AC/DC/daily/20260517/20260820"}
Bosnia_and_Herzegovina Template:List_of_settlements_in_the_Federation_of_Bosnia_and_Herzegovina/top 404 {"detail":"invalid route","method":"get","status":404,"title":"Not Found","type":"about:blank","uri":"/metrics/pageviews/per-article/en.wikipedia.org/all-access/all-agents/Template:List_of_settlements_in_the_Federation_of_Bosnia_and_Herzegovina/top/daily/20260517/20260820"}
Belgium Roman/Red 404 {"detail":"invalid route","method":"get","status":404,"title":"Not Found","type":"about:blank","uri":"/metrics/pageviews/per-article/en.wikipedia.org/all-access/all-agents/Roman/Red/daily/20260517/20260820"}


,country,subcategory_title_fixed,article_title,date,views
0,Paraguay,Paraguay-related lists,Outline_of_Paraguay,20260517,9
1,Paraguay,Paraguay-related lists,Outline_of_Paraguay,20260518,9
2,Paraguay,Paraguay-related lists,Outline_of_Paraguay,20260519,10
3,Paraguay,Paraguay-related lists,Outline_of_Paraguay,20260520,11
4,Paraguay,Paraguay-related lists,Outline_of_Paraguay,20260521,18
...,...,...,...,...,...
110159,Belgium,Images of Belgium,File:Marthe_Donas_-_Still_Life_with_Bottle_and...,20260729,2
110160,Belgium,Images of Belgium,File:Marthe_Donas_-_Still_Life_with_Bottle_and...,20260730,2
110161,Belgium,Images of Belgium,File:Marthe_Donas_-_Still_Life_with_Bottle_and...,20260807,2
110162,Belgium,Images of Belgium,File:Marthe_Donas_-_Still_Life_with_Bottle_and...,20260810,1


In [46]:
## Now save as .csv, shape is (110164x5)
article_views_df.to_csv('../data/article_views.csv', index=False)

In [76]:
## Now append analytics to article_views_df

## First calculate mean views, standard deviation per article over each cup window.
#mean_views = article_views_df.groupby(['country', 'subcategory_title_fixed', 'article_title'])['views'].agg({}).reset_index().rename(columns={'views': 'mean_views'})
regional_search_interest_df[['cultural_interest_1','cultural_interest_2','match_interest_1','match_interest_2']].sum()

cultural_interest_1    34855
cultural_interest_2       24
match_interest_1        4905
match_interest_2         720
dtype: int64

In [50]:
## Now get data on each country's main page views during each window
country_view_data = []

## Call function 4
country_daily_views_df = get_country_wikipedia_pageviews(cup_windows, headers)
country_daily_views_df

country_daily_views_df.to_csv('../data/country_article_views.csv', index=False)

,country,article_title,date,views
0,Australia,Australia,20260517,18848
1,Australia,Australia,20260518,17349
2,Australia,Australia,20260519,14740
3,Australia,Australia,20260520,14066
4,Australia,Australia,20260521,13189
...,...,...,...,...
475,Turkey,Turkey,20260816,10865
476,Turkey,Turkey,20260817,11153
477,Turkey,Turkey,20260818,10818
478,Turkey,Turkey,20260819,10596


In [58]:
## Now get the national football team's wikipedia pageviews during the same window, call function 5
country_football_data = []

country_football_views_df = get_football_wikipedia_pageviews(cup_windows, headers)
country_football_views_df

country_football_views_df.to_csv('../data/football_views.csv')

,country,article_title,date,views
0,Australia,Australia National Football Team,20260517,10
1,Australia,Australia National Football Team,20260518,20
2,Australia,Australia National Football Team,20260519,20
3,Australia,Australia National Football Team,20260520,34
4,Australia,Australia National Football Team,20260521,37
...,...,...,...,...
475,Turkey,Turkey National Football Team,20260816,1498
476,Turkey,Turkey National Football Team,20260817,1497
477,Turkey,Turkey National Football Team,20260818,1526
478,Turkey,Turkey National Football Team,20260819,1440


In [86]:
## Now analyze google search trends. I reckon that just selecting keywords will do. I used AI to help me code this part, since I was
## grossly unfamiliar with pytrends. Learning something new!
pytrend = TrendReq(hl='en-US', tz=360)
## Import host city data in geoName format for API to read
host_cities = ['Atlanta GA', 'Boston MA', 'Dallas TX', 'Houston TX', 'Kansas-City MO', 'Los-Angeles CA', 'Miami FL', 'New-York NY', 'Philidelphia PA',
              'Seattle WA', 'San-Francisco CA']

## Map to readable geo codes. These are publicly available so I just took them from online https://github.com/epfl-dlab/GoogleTrendsAnchorBank
host_codes = ['US-GA-524', 'US-MA-506', 'US-TX-623', 'US-TX-618', 'US-MO-616','US-CA-803', 'US-FL-528', 'US-NY-501', 'US-PA-504', 
             'US-WA-819', 'US-CA-807']

## Define empty dataframe
regional_search_results = []

regional_search_interest_df = regional_search_interest(cup_windows, host_codes)
regional_search_interest_df

Bosnia_and_Herzegovina US-FL-528 The request failed: Google returned a response with code 429
Turkey US-FL-528 The request failed: Google returned a response with code 429


,date,cultural_interest_1,cultural_interest_2,match_interest_1,match_interest_2,isPartial,country,geo_code
0,2026-05-17,0,0,0,0,False,Australia,US-GA-524
1,2026-05-18,0,0,0,0,False,Australia,US-GA-524
2,2026-05-19,0,0,0,0,False,Australia,US-GA-524
3,2026-05-20,0,0,0,0,False,Australia,US-GA-524
4,2026-05-21,0,0,0,0,False,Australia,US-GA-524
...,...,...,...,...,...,...,...,...
5083,2026-08-16,0,0,0,0,False,Turkey,US-CA-807
5084,2026-08-17,0,0,0,0,False,Turkey,US-CA-807
5085,2026-08-18,0,0,0,0,False,Turkey,US-CA-807
5086,2026-08-19,0,0,0,0,False,Turkey,US-CA-807


In [89]:
regional_search_interest_df['date'] = pd.to_datetime(regional_search_interest_df['date'])

regional_search_weekly_df = regional_search_interest_df.set_index('date').groupby(['country', 'geo_code', pd.Grouper(freq='W')])[[ 'cultural_interest_1', 'cultural_interest_2', 'match_interest_1', 'match_interest_2']].sum().reset_index()

regional_search_weekly_df

regional_search_weekly_df.to_csv('../data/regional_search_interest.csv', index=False)

,country,geo_code,date,cultural_interest_1,cultural_interest_2,match_interest_1,match_interest_2
0,Australia,US-CA-803,2026-05-17,2,0,0,0
1,Australia,US-CA-803,2026-05-24,14,0,6,0
2,Australia,US-CA-803,2026-05-31,13,0,18,14
3,Australia,US-CA-803,2026-06-07,14,1,19,3
4,Australia,US-CA-803,2026-06-14,16,0,128,16
...,...,...,...,...,...,...,...
790,Turkey,US-WA-819,2026-07-26,0,0,0,0
791,Turkey,US-WA-819,2026-08-02,0,0,0,0
792,Turkey,US-WA-819,2026-08-09,0,0,0,0
793,Turkey,US-WA-819,2026-08-16,0,0,0,0


In [54]:
## I did not end up doing this but it was kind of a cool path to go down
## Part 3: Access census data

## Define census api key, please don't take mine!
##census_api = '9fdf899b3427f195717633b957799dd41ae17fdf'
##census_api_key = os.environ.get('CENSUS_API_KEY', f'{census_api}')

## I don't feel like writing a function to call for specific country codes within the B05006 table. I will pull these manually instead 
## for the 2022 and 2024 sets

##countries_2022 = ['Wales', 'England', 'Iran', 'Netherlands']
##codes_2022 = ['B05006_009E', 'B05006_010E', 'B05006_061E', 'B05006_018E']

## 2024 is the most recent available census data, reasonable to take given migration patterns on the order of 5-10 years+.
##countries_2024 = ['Paraguay', 'Australia', 'Turkey', 'Bosnia_and_Herzegovina', 'Belgium']
##codes_2024 = ['B05006_175E', 'B05006_132E', 'B05006_090E', 'B05006_031E', 'B05006_015E']

##census_table_2022 = pd.DataFrame({"country": countries_2022, "code": codes_2022})
##census_table_2024 = pd.DataFrame({"country": countries_2024, "code": codes_2024})

##census_table_2022
##census_table_2024

## Now pull diaspora data. We just want the city with the highest fraction of diaspora during the world cup.
## We must also pull from the city's total population (located in B01003_001E). 
## I will write a user-defined function that is generalizable to my tables


#def diasporaFraction(code, year):
 #   call_link = f'https://api.census.gov/data/{year}/acs/acs5'
#
 #   census_all_data = []
  #  for x in code:
   #     params = {
    #        'get': f'NAME,{x},B01003_001E',
     #       'for': 'place:*',
      #      'key': census_api_key
       # }
#
 #       ## Format to json data immediately, I think I understand requests.get() well enough
  #      formatted_response = requests.get(call_link, params=params).json()
#
        ## Convert to df
 #       census_formatted_data = pd.DataFrame(formatted_response[1:], columns=formatted_response[0])
#
 #       census_formatted_data[x] = pd.to_numeric(census_formatted_data[x], errors='coerce')
  #      census_formatted_data['B01003_001E'] = pd.to_numeric(census_formatted_data['B01003_001E'], errors='coerce')
   #     census_formatted_data['frac_diaspora'] = census_formatted_data[x] / census_formatted_data['B01003_001E']
        ## Sort by fraction of diaspora
    #    census_formatted_data = census_formatted_data.sort_values('frac_diaspora', ascending=False)
        ## Now filter for large cities, I got a result that said Buttzville, NJ contained the highest fraction
        ## of Dutch people. The population of Buttzville is 103.
     #   census_formatted_data = census_formatted_data[census_formatted_data['B01003_001E'] > 50000].head(1)
      #  census_all_data.append(census_formatted_data)

    #return pd.concat(census_all_data, ignore_index=True)
    
#diaspora2022 = diasporaFraction(codes_2022, '2022')

#diaspora2024 = diasporaFraction(codes_2024, '2024')
## Examine datasets to find regions we need to extract from search api

#diaspora2022
#diaspora2024

#diaspora2022.to_csv('./data/diaspora_2022.csv', index=False)
#diaspora2024.to_csv('./data/diaspora_2024.csv', index=False)

,NAME,B05006_009E,B01003_001E,state,place,frac_diaspora,B05006_010E,B05006_061E,B05006_018E
0,"Laguna Niguel city, California",662.0,64259,06,39248,0.010302,NaN,NaN,NaN
1,"Santa Monica city, California",NaN,92168,06,70000,0.007063,651.0,NaN,NaN
2,"Glendale city, California",NaN,194512,06,30000,0.138135,NaN,26869.0,NaN
3,"Horizon West CDP, Florida",NaN,58595,12,32610,0.010632,NaN,NaN,623.0


,NAME,B05006_175E,B01003_001E,state,place,frac_diaspora,B05006_132E,B05006_090E,B05006_031E,B05006_015E
0,"Weston city, Florida",512.0,68837,12,76582,0.007438,NaN,NaN,NaN,NaN
1,"Santa Monica city, California",NaN,91169,06,70000,0.004640,423.0,NaN,NaN,NaN
2,"Alpharetta city, Georgia",NaN,66855,13,01696,0.010605,NaN,709.0,NaN,NaN
3,"Utica city, New York",NaN,64217,36,76540,0.039927,NaN,NaN,2564.0,NaN
4,"Deerfield Beach city, Florida",NaN,88093,12,16725,0.004030,NaN,NaN,NaN,355.0
